# 🌊 Phytoplankton Detection with YOLO26
### Hybrid Runner: Kaggle GPU (Full Training) & Local Machine (Debugging/Test)

Notebook ini dirancang secara adaptif:
- **Di Kaggle**: Mengkloning kode dari GitHub, menghubungkan dataset dari `/kaggle/input`, dan melatih model dengan GPU (100 epoch).
- **Di Komputer Lokal**: Berjalan langsung tanpa perlu klon, menggunakan CPU/GPU lokal dengan mode debug cepat (1-2 epoch) untuk pengujian alur.

---

## 1. Deteksi Lingkungan (Environment & Hardware Check)
Mengecek apakah notebook berjalan di Kaggle atau di lokal, serta mendeteksi akselerator CUDA/GPU.

In [ ]:
import os
import sys
from pathlib import Path
import torch

# Deteksi lingkungan otomatis
IS_KAGGLE = Path("/kaggle/working").exists() or Path("/kaggle/input").exists()
ENV_NAME = "Kaggle" if IS_KAGGLE else "Local Machine"

print(f"[ENV] Berjalan di: {ENV_NAME}")
print(f"[PY]  Python version: {sys.version.split()[0]}")
print(f"[TORCH] PyTorch version: {torch.__version__}")
print(f"[CUDA]  CUDA Tersedia: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"[DEVICE] Nama GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[DEVICE] Menggunakan CPU (Aman untuk local debugging)")

## 2. Install Dependencies
Menginstal library Ultralytics dan dependensi computer vision pendukung.

In [ ]:
# Install dependensi yang dibutuhkan
!pip install -q --upgrade ultralytics
!pip install -q pyyaml opencv-python pillow tqdm matplotlib seaborn

## 3. Setup Repositori GitHub
- **Jika di Kaggle**: Otomatis mengkloning repositori GitHub Anda ke `/kaggle/working/` (atau `git pull` jika sudah ada).
- **Jika di Lokal**: Otomatis dilewati karena Anda sudah berada di dalam folder proyek.

In [ ]:
# GANTI DENGAN URL REPOSITORY GITHUB ANDA SETELAH PUSH
GITHUB_REPO_URL = "https://github.com/mahruzaley-pixel/phytoplankton_detection.git"
REPO_NAME = "phytoplankton_detection"

if IS_KAGGLE:
    working_dir = Path("/kaggle/working")
    project_dir = working_dir / REPO_NAME
    
    if not project_dir.exists():
        print(f"[GIT] Mengkloning {GITHUB_REPO_URL} ke {project_dir}...")
        !git clone {GITHUB_REPO_URL} {project_dir}
    else:
        print(f"[GIT] Repositori sudah ada. Mengambil pembaruan terbaru (git pull)...")
        %cd {project_dir}
        !git pull
    
    %cd {project_dir}
    print(f"[DIR] Direktori kerja Kaggle saat ini: {os.getcwd()}")
else:
    # Di komputer lokal, gunakan direktori repo saat ini
    project_dir = Path.cwd()
    print(f"[GIT] Berjalan di lokal, melewati kloning. Folder proyek: {project_dir}")

## 4. Deteksi & Sambungkan Dataset
Mencari lokasi dataset (apakah dari `/kaggle/input/` atau dari folder lokal `sub_dataset`/`dataset`).

In [ ]:
import glob

raw_dataset_path = None

if IS_KAGGLE:
    # Cari di /kaggle/input (dataset yang diunggah ke Kaggle)
    possible_paths = glob.glob("/kaggle/input/**/Annotations", recursive=True)
    if possible_paths:
        raw_dataset_path = Path(possible_paths[0]).parent
        print(f"[FOUND] Kaggle Dataset ditemukan di: {raw_dataset_path}")

if raw_dataset_path is None:
    # Fallback ke folder lokal (sub_dataset yang sudah difilter atau dataset asli)
    for candidate in [Path("sub_dataset"), Path("dataset"), project_dir / "sub_dataset", project_dir / "dataset"]:
        if candidate.exists() and (candidate / "Annotations").exists():
            raw_dataset_path = candidate.resolve()
            print(f"[FOUND] Dataset lokal ditemukan di: {raw_dataset_path}")
            break

if raw_dataset_path is None:
    raise FileNotFoundError("Dataset tidak ditemukan! Hubungkan dataset di Kaggle atau letakkan di sub_dataset/")

# Deteksi konfigurasi kelas (prioritaskan classes.yaml milik sub_dataset)
if (raw_dataset_path / "classes.yaml").exists():
    classes_config_path = raw_dataset_path / "classes.yaml"
else:
    classes_config_path = Path("configs/classes.yaml")

print(f"[CONFIG] Menggunakan konfigurasi kelas: {classes_config_path}")
print("  Jumlah XML Annotations:", len(list((raw_dataset_path / 'Annotations').glob('*.xml'))))
print("  Jumlah JPEGImages     :", len(list((raw_dataset_path / 'JPEGImages').glob('*.*'))))

## 5. Konfigurasi Mode Pelatihan
Pilih mode eksekusi:
- `"debug"`: Cepat untuk pengujian lokal (2 epoch, resolusi 416, batch kecil).
- `"full"`: Pelatihan penuh di Kaggle GPU (100 epoch, resolusi 800, batch 16).

In [ ]:
# Opsi: 'auto', 'debug', atau 'full'
# Opsi: "full" (Full Training 100 Epoch) atau "debug" (Uji Coba Cepat 2 Epoch)
RUN_MODE = "full"  # Default "full" untuk full training di Kaggle
IS_DEBUG = (RUN_MODE == "debug")

if IS_DEBUG:
    print("⚙️ MODE: DEBUG / LOCAL TEST (Verifikasi Cepat)")
    EPOCHS = 2
    BATCH_SIZE = 4
    IMGSZ = 416
    MODEL_NAME = "yolo26n.pt"
    PREP_MODE = "mini"
    MINI_SAMPLES = 30
    DEVICE = "0" if torch.cuda.is_available() else "cpu"
    DATA_DIR = "data/yolo_debug"
    EXP_NAME = "yolo26_debug"
else:
    print("🚀 MODE: FULL TRAINING (Kaggle GPU)")
    EPOCHS = 100
    BATCH_SIZE = 16
    IMGSZ = 800
    MODEL_NAME = "yolo26s.pt"  # atau 'yolo26m.pt'
    PREP_MODE = "full"
    MINI_SAMPLES = 40
    DEVICE = "0" if torch.cuda.is_available() else "cpu"
    DATA_DIR = "data/yolo"
    EXP_NAME = "yolo26_phytoplankton_full"

print(f"  Epochs : {EPOCHS}")
print(f"  Batch  : {BATCH_SIZE}")
print(f"  Imgsz  : {IMGSZ}")
print(f"  Device : {DEVICE}")

## 6. Konversi Dataset ke Format Standar YOLO26
Menjalankan script modular `prepare_data.py` untuk mengonversi anotasi VOC ke YOLO normalisasi dan membaginya ke train/val/test split.

In [ ]:
!python prepare_data.py \
    --raw_dir "{raw_dataset_path}" \
    --output_dir "{DATA_DIR}" \
    --mode {PREP_MODE} \
    --mini_samples {MINI_SAMPLES} \
    --classes_config "{classes_config_path}"

yolo_data_yaml = Path(DATA_DIR) / "data.yaml"
if not yolo_data_yaml.exists():
    raise FileNotFoundError(f"File {yolo_data_yaml} gagal dibuat. Periksa error di atas!")
print(f"\n[OK] Config dataset YOLO siap di: {yolo_data_yaml}")

## 7. Jalankan Training YOLO26
Melatih arsitektur YOLO26 dengan fitur NMS-Free End-to-End inference dan optimizer yang dioptimalkan untuk objek mikroskopik.

In [ ]:
!python train.py \
    --data "{yolo_data_yaml}" \
    --model {MODEL_NAME} \
    --epochs {EPOCHS} \
    --batch {BATCH_SIZE} \
    --imgsz {IMGSZ} \
    --device {DEVICE} \
    --name {EXP_NAME}

## 8. Evaluasi Model pada Test Set
Menghitung metrik performa: Precision, Recall, mAP@50, dan mAP@50-95 pada data uji (test split).

In [ ]:
candidates = list(Path("runs").rglob(f"*{EXP_NAME}*/weights/best.pt"))
if not candidates:
    candidates = list(Path("runs").rglob("best.pt"))
best_weights = candidates[0] if candidates else Path(f"runs/train/{EXP_NAME}/weights/best.pt")

print(f"[EVAL] Menggunakan checkpoint: {best_weights}")

!python val.py \
    --weights "{best_weights}" \
    --data "{yolo_data_yaml}" \
    --split test \
    --imgsz {IMGSZ} \
    --device {DEVICE}

## 9. Visualisasi Grafik Metrik & Hasil Training

In [ ]:
from IPython.display import Image, display

exp_dirs = [p for p in Path("runs").rglob(EXP_NAME) if p.is_dir() and (p / "weights").exists()]
exp_dir = exp_dirs[0] if exp_dirs else Path(f"runs/train/{EXP_NAME}")
metric_plots = ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]

for plot in metric_plots:
    p = exp_dir / plot
    if p.exists():
        print(f"\n=== {plot} ===")
        display(Image(str(p)))
    else:
        print(f"[INFO] {plot} belum tersedia (hanya dibuat setelah beberapa epoch).")

## 10. Inferensi & Visualisasi Bounding Box pada Citra Test

In [ ]:
test_images_dir = Path(DATA_DIR) / "images" / "test"
predict_save_dir = Path(f"runs/predict_{EXP_NAME}")

!python predict.py \
    --weights "{best_weights}" \
    --source "{test_images_dir}" \
    --conf 0.15 \
    --imgsz {IMGSZ} \
    --device {DEVICE} \
    --save_dir "{predict_save_dir}"

# Tampilkan 4 contoh hasil deteksi
predicted_images = list(predict_save_dir.glob("*.jpg"))[:4]
if predicted_images:
    print(f"\nMenampilkan {len(predicted_images)} sampel hasil deteksi:")
    for img_p in predicted_images:
        display(Image(str(img_p)))

## 11. Export & Download Checkpoint Terbaik (`best.pt`)
Mengompresi bobot model terbaik beserta grafik hasil evaluasi ke dalam satu file ZIP yang dapat di-download langsung dari Kaggle.

In [ ]:
import zipfile
from IPython.display import FileLink

if IS_KAGGLE:
    export_zip_path = Path("/kaggle/working/yolo26_phytoplankton_results.zip")
else:
    export_zip_path = Path("runs/yolo26_results.zip")

export_zip_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Mengompresi hasil ke {export_zip_path}...")
with zipfile.ZipFile(export_zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    runs_dir = Path("runs")
    if runs_dir.exists():
        for root, _, files in os.walk(runs_dir):
            for f in files:
                full_p = Path(root) / f
                if full_p.suffix == ".zip":
                    continue
                rel_p = full_p.relative_to(runs_dir)
                zipf.write(full_p, arcname=str(Path("runs") / rel_p))
    if yolo_data_yaml.exists():
        zipf.write(yolo_data_yaml, arcname=yolo_data_yaml.name)
    if classes_config_path.exists():
        zipf.write(classes_config_path, arcname=classes_config_path.name)

print(f"\n[SELESAI] File hasil siap: {export_zip_path} ({export_zip_path.stat().st_size / (1024*1024):.2f} MB)")
FileLink(str(export_zip_path))